# 阶段四：合成 Reward 训练闭环验证

本 Notebook 不读取行情、Barra 或行业数据。它验证状态编码 → Transformer → DAG 采样 → 合成 Reward → TB Loss → backward → optimizer.step，并展示训练监控和检查点恢复。

## 1. 环境与项目导入检查

**测试目标**：确认 Notebook 从项目根目录导入 `factor_gfn`，并使用当前 Python 环境中的 PyTorch。该单元只做路径和依赖检查，不开始训练。

**如何看结果**：

- `project` 应为 `D:\\实习\\Gflownet因子挖掘`；
- `torch` 应输出已安装版本；
- 如果出现 `ModuleNotFoundError`，通常是内核选错或环境依赖未安装；
- 可额外运行 `import sys; print(sys.executable)`，应指向项目内 `.venv\\python.exe`。

In [ ]:
from pathlib import Path
from dataclasses import asdict
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from factor_gfn.gfn import (
    GFNConfig, GFNTrainer, ModelConfig, SamplingConfig,
    SearchSpaceConfig, SyntheticRewardProvider, TrainingConfig,
    write_run_metadata,
)
print('project:', PROJECT_ROOT)
print('torch:', torch.__version__, '| device: cpu')

## 2. 创建小型训练配置与合成 Reward

**测试目标**：创建适合 CPU 快速验证的小型 Transformer、搜索空间和 Trainer，并保存训练前参数作为对照。这里使用合成 Reward，不读取真实行情。

合成目标偏好包含 `close` 和 `ts_mean` 的表达式，同时轻微惩罚节点数；它只用于证明训练链路能工作，不代表真实因子评价。

**如何看结果**：

- 单元格末尾应显示 `optimizer_eps` 和 `reward_floor` 两项；
- 两者数值均为 `1e-8`，但说明文字必须不同；
- 本单元不应产生警告或异常；初始 `logZ` 被记录为 0。

In [ ]:
config = GFNConfig(
    search_space=SearchSpaceConfig(max_depth=3, max_nodes=7),
    model=ModelConfig(d_model=32, num_heads=4, num_layers=1, dim_feedforward=64, dropout=0.0),
    sampling=SamplingConfig(temperature=1.0, greedy=False),
    training=TrainingConfig(
        batch_size=8, learning_rate=1e-3, log_z_learning_rate=1e-2,
        max_steps=50, model_gradient_clip_norm=1.0, log_z_gradient_clip_norm=1.0,
        optimizer_eps=1e-8, max_sampling_multiplier=10, seed=2026,
    ),
)
trainer = GFNTrainer(config, SyntheticRewardProvider(), device='cpu')
initial_model = {name: value.detach().clone() for name, value in trainer.model.state_dict().items()}
initial_log_z = float(trainer.tb_loss.log_z.detach())
trainer.run_metadata()['parameter_semantics']

## 3. 运行 20 步真实参数更新

**测试目标**：真正执行20次“DAG采样 → 合成 Reward → TB Loss → backward → 梯度裁剪 → Adam step”。CPU通常需要约1分钟。

**关键列与通过标准**：

- `loss`：必须始终有限；短训练中不要求单调下降，因为每步采样的表达式不同；
- `log_z`：应逐步变化，说明可学习总流量参与更新；
- `reward_mean`：当前 batch 的平均合成奖励；
- `expression_unique_rate`：越接近1表示 batch 内重复表达式越少；
- `policy_entropy_mean`：联合槽位/Token策略原始熵，会受当前合法动作数影响；
- `policy_entropy_normalized_mean`：按 `H/log(K_legal_joint)` 归一化的熵，更适合跨状态比较；长期突然接近0可能表示探索坍塌；
- `batch_rejection_rate`：合成 Reward 下应为0；真实 Reward 阶段才可能升高；
- `effective_batch_size`：应始终为8；
- `illegal_action_rate`：必须始终为0。

单元格末尾的断言会自动检查有限 Loss、零非法动作率和未跳过更新；失败时会直接报错。

In [ ]:
stats = trainer.train(20)
history = pd.DataFrame([asdict(item) for item in stats])
display(history[[
    'step', 'loss', 'log_z', 'reward_mean', 'expression_unique_rate',
    'trajectory_length_mean', 'policy_entropy_mean', 'policy_entropy_normalized_mean',
    'batch_rejection_rate', 'effective_batch_size', 'illegal_action_rate',
]])
assert np.isfinite(history['loss']).all()
assert (history['illegal_action_rate'] == 0).all()
assert history['policy_entropy_normalized_mean'].dropna().between(0, 1).all()
assert not history['skipped_update'].any()

## 4. 可视化训练过程

**测试目标**：把表格中的核心指标画成曲线，帮助发现数值爆炸、探索坍塌或采样复杂度异常。

**如何看结果**：

- `TB Loss` 可以波动，但不能出现 NaN、Inf 或持续爆炸；
- `Learnable logZ` 应连续变化，不应完全固定；
- `Unique Rate` 长期接近0表示重复表达式过多；
- `Entropy` 快速降到接近0可能表示策略过早变成确定性；
- `Length` 显示表达式复杂度，长期顶到 `max_nodes=7` 时需检查长度偏好。

20步只用于连通性检查，不足以据此判断模型是否已经收敛。

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 7))
history.plot(x='step', y='loss', ax=axes[0, 0], title='TB Loss')
history.plot(x='step', y='log_z', ax=axes[0, 1], title='Learnable logZ')
history.plot(x='step', y=['reward_mean', 'expression_unique_rate'], ax=axes[1, 0], title='Reward / Unique Rate')
history.plot(x='step', y=['policy_entropy_mean', 'trajectory_length_mean'], ax=axes[1, 1], title='Entropy / Length')
plt.tight_layout()
plt.show()

## 5. 确认模型参数和 logZ 确实更新

**测试目标**：将训练后的 Transformer 参数与训练前副本逐项比较，并核对 `logZ` 和优化器步数。仅有 Loss 数值还不能证明 `optimizer.step()` 真正生效，因此需要这一项。

**如何看结果**：

- `changed_model_tensors` 必须大于0；
- `final_log_z` 必须不同于 `initial_log_z=0`；
- `optimizer_steps` 应为20；
- 任一条件不满足都会触发断言，说明梯度链路或参数更新存在问题。

In [ ]:
changed = [
    name for name, value in trainer.model.state_dict().items()
    if not torch.equal(value, initial_model[name])
]
parameter_summary = {
    'changed_model_tensors': len(changed),
    'total_model_tensors': len(initial_model),
    'initial_log_z': initial_log_z,
    'final_log_z': float(trainer.tb_loss.log_z.detach()),
    'optimizer_steps': trainer.optimizer_step,
}
assert changed
assert parameter_summary['final_log_z'] != initial_log_z
parameter_summary

## 6. 保存、恢复并继续训练

**测试目标**：把模型、`logZ`、优化器、配置指纹、随机状态和统计历史保存到检查点，再用新 Trainer 恢复，并继续训练3步。文件写入 `tmp/synthetic_runs/`，不会进入Git。

**如何看结果**：

- 恢复后模型参数和 `logZ` 必须与保存时完全一致，否则断言失败；
- 输出路径应指向 `tmp/synthetic_runs/manual_validation.pt` 和元数据JSON；
- `restored step` 与 `optimizer step` 最终都应为23，即原20步加续跑3步；
- 最后的三行统计必须保持有限 Loss、有效 batch 和零非法动作率。

自动化测试还额外比较了连续训练与中途保存恢复训练的最终参数、优化器及随机状态；本单元主要用于人工查看。

In [ ]:
run_dir = PROJECT_ROOT / 'tmp' / 'synthetic_runs'
checkpoint_path = run_dir / 'manual_validation.pt'
metadata_path = run_dir / 'manual_validation_metadata.json'
trainer.save_checkpoint(checkpoint_path)
write_run_metadata(metadata_path, trainer)

restored = GFNTrainer(config, SyntheticRewardProvider(), device='cpu')
restored.load_checkpoint(checkpoint_path)
for name, value in trainer.model.state_dict().items():
    assert torch.equal(value, restored.model.state_dict()[name]), name
assert torch.equal(trainer.tb_loss.log_z.detach(), restored.tb_loss.log_z.detach())
continued = restored.train(3)
print('checkpoint:', checkpoint_path)
print('metadata:', metadata_path)
print('restored step:', restored.step, '| optimizer step:', restored.optimizer_step)
pd.DataFrame([asdict(item) for item in continued])